# MetaQA Full EDA

This notebook analyzes the MetaQA knowledge graph and QA splits used by the Pocket GraphRAG project. It covers dataset size, answer distributions, question lengths, topic entity frequency, node/edge statistics, relation frequencies, degree distributions, sampled path lengths, and question-type signals across 1-hop, 2-hop, and 3-hop data.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src' / 'eda'))

from metaqa_analysis import analyze, set_plot_style

set_plot_style()
RAW_DIR = PROJECT_ROOT / 'src' / 'data' / 'raw'
analysis = analyze(RAW_DIR)

qa_df = analysis['qa_df']
graph = analysis['graph']
relations = analysis['relations']
degrees = analysis['degrees']
print(f'Loaded {len(qa_df):,} QA examples')
print(f'Graph nodes: {graph.number_of_nodes():,}')
print(f'Graph edges: {graph.number_of_edges():,}')

## 1. Dataset-Level EDA

This section checks whether the hop splits are balanced, how many answers each question has, how long questions are, and which topic entities dominate the dataset.

In [ ]:
analysis['dataset_summary']

In [ ]:
from metaqa_analysis import plot_question_counts
plot_question_counts(qa_df);

In [ ]:
from metaqa_analysis import plot_answer_counts
plot_answer_counts(qa_df);

In [ ]:
from metaqa_analysis import plot_question_lengths
plot_question_lengths(qa_df);

In [ ]:
from metaqa_analysis import plot_top_topic_entities
plot_top_topic_entities(qa_df, top_n=20);

## 2. Question-Type Signals

The question templates often imply relation demand. This approximate view helps identify whether one relation family dominates a hop level.

In [ ]:
analysis['question_types'].head(30)

In [ ]:
from metaqa_analysis import plot_question_types
plot_question_types(analysis['question_types']);

## 3. Knowledge Graph Overview

This section reports graph-level statistics: nodes, edges, relation count, density, connected components, and largest component size.

In [ ]:
analysis['graph_summary']

## 4. Edge-Level EDA

Relation frequency shows which relation types dominate the knowledge graph. This matters because retrievers and models may overfit to common relations.

In [ ]:
relations

In [ ]:
from metaqa_analysis import plot_relation_counts
plot_relation_counts(relations);

## 5. Node-Level EDA

Degree distributions reveal hub entities. High-degree movie or actor nodes can inflate performance if test questions concentrate around popular entities.

In [ ]:
degrees.describe()

In [ ]:
from metaqa_analysis import plot_degree_distribution
plot_degree_distribution(degrees);

In [ ]:
from metaqa_analysis import plot_top_degree_entities
plot_top_degree_entities(degrees, 'degree', top_n=20);

In [ ]:
from metaqa_analysis import plot_top_degree_entities
plot_top_degree_entities(degrees, 'in_degree', top_n=20);

In [ ]:
from metaqa_analysis import plot_top_degree_entities
plot_top_degree_entities(degrees, 'out_degree', top_n=20);

In [ ]:
from metaqa_analysis import plot_in_vs_out_degree
plot_in_vs_out_degree(degrees);

## 6. Graph-Level Path Analysis

The sampled shortest-path distribution gives a practical estimate of graph navigability without computing all-pairs shortest paths over the full graph.

In [ ]:
analysis['sampled_paths'].describe()

In [ ]:
from metaqa_analysis import plot_shortest_paths
plot_shortest_paths(analysis['sampled_paths']);

## 7. Important Takeaways to Write in the Report

- Multi-answer questions make exact match harsh; report F1 and Hits@1/EM.
- High-degree hubs may inflate performance for common entities and hurt rare entities.
- Relation imbalance can reveal which reasoning patterns the model sees most often.
- 3-hop questions should be analyzed separately because retrieval and subgraph coverage usually degrade with hop depth.
- Use graph coverage and Recall@K to determine whether failures come from retrieval/subgraph extraction or answer generation.